In [2]:
!pip install --upgrade pip

# YOLOv8 (use latest stable)
!pip install "ultralytics==8.*"

# OpenCV
!pip install "opencv-python-headless>=4.6.0"

# Torch (adjust for your CUDA)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# OC-SORT tracker packaged
!pip install ocsort

# Required by some trackers
!pip install lap filterpy

# Optional: helper visualization tools
!pip install supervision

Looking in indexes: https://download.pytorch.org/whl/cu118
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.5 MB ? eta -:--:--
   ----- ---------------------------------- 0.8/5.5 MB 4.3 MB/s eta 0:00:02
   ------------------- -------------------- 2.6/5.5 MB 5.9 MB/s eta 0:00:01
   -------------------------------- ------- 4.5/5.5 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 6.0 MB/s  0:00:01
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 6.8 MB/s eta 0:06:57
   ---------------------------------------- 0.0/2.8 GB 6.7 MB/s eta 0:07:02
   ---------------------------------------- 0.0/2.8 GB 6.7 MB/s eta 0:07:01
   ---------------------------------------- 0.0/2.8 GB 6.7

  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [26 lines of output]
  <string>:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  Partial import of lap during the build process.
  Traceback (most recent call last):
    File "<string>", line 127, in get_numpy_status
  ModuleNotFoundError: No module named 'numpy'
  Traceback (most recent call last):
    File "C:\Users\avnis\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
      ~~~~^^
    File "C:\Users\avnis\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                              

# ✅ OPTIMIZED INFERENCE CELLS (APPENDED)
The following cells were added automatically to improve realtime performance:
- Async capture
- Batching
- FP16 / TensorRT fallback
- Minimal per-frame overhead

You can run these cells or copy them into the inference section of your notebook.

In [1]:
# %%
# IMPORTS & DEVICE CHECK
import os
import time
from threading import Thread
from queue import Queue, Empty
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics import YOLO

print('Torch CUDA available:', torch.cuda.is_available())
print('Torch version:', torch.__version__)


Torch CUDA available: False
Torch version: 2.9.1+cpu


In [8]:
# %%
# CONFIGURATION (tune these)
CFG = {
    'SOURCE': 0,                # 0 for webcam, or 'video.mp4', or 'rtsp://...'
    'MODEL_PRETRAINED': 'yolov8n.pt',  # default pre-trained weight
    'MODEL_CUSTOM': './checkpoints/yolov8m_advanced.pt',  # uncomment to use custom
    'USE_CUSTOM': False,
    'IMG_SIZE': 640,            # base model size (try 480 or 360 for speed)
    'BATCH': 2,                 # frames per batch (1 = no batching). Increase for faster GPUs
    'CONF': 0.35,
    'IOU': 0.45,
    'DEVICE': 0 if torch.cuda.is_available() else 'cpu',
    'USE_TRT': True,            # try to use TensorRT engine if exported/available
    'TRT_PATH': None,           # path to .engine file if you have one, else None
    'FP16': True,               # use fp16 if supported
    'SHOW': True,               # use cv2.imshow (desktop) — False for headless
    'USE_TRACKER': True,        # enable OC-SORT (optional)
}


In [3]:
# %%
# MODEL LOADING with TensorRT / ONNX fallback
from pathlib import Path

def load_model(cfg):
    # Priority: TRT engine (if present and requested) -> custom PT -> pretrained PT
    if cfg['USE_TRT'] and cfg['TRT_PATH'] and Path(cfg['TRT_PATH']).exists():
        print('Loading TensorRT engine:', cfg['TRT_PATH'])
        model = YOLO(cfg['TRT_PATH'])
        return model

    if cfg['USE_CUSTOM'] and Path(cfg['MODEL_CUSTOM']).exists():
        print('Loading custom .pt:', cfg['MODEL_CUSTOM'])
        model = YOLO(cfg['MODEL_CUSTOM'])
        return model

    print('Loading pretrained model:', cfg['MODEL_PRETRAINED'])
    model = YOLO(cfg['MODEL_PRETRAINED'])
    return model

model = load_model(CFG)
# Prefer using half precision if GPU available and model supports it
if CFG['FP16'] and torch.cuda.is_available():
    try:
        model.model.half()
        print('Set model to fp16')
    except Exception:
        pass

# Helpful mapping for class names
try:
    NAMES = model.model.names if hasattr(model, 'model') else model.names
except Exception:
    NAMES = None

print('Model loaded, names sample:', list(NAMES.items())[:5] if NAMES else 'N/A')


Loading pretrained model: yolov8m.pt
Model loaded, names sample: [(0, 'person'), (1, 'bicycle'), (2, 'car'), (3, 'motorcycle'), (4, 'airplane')]


In [4]:
# %%
# ASYNC VIDEO CAPTURE
class FastVideoCapture:
    def __init__(self, source=0, width=None, height=None, queue_size=8):
        self.source = source
        if isinstance(source, int):
            self.cap = cv2.VideoCapture(source)
        else:
            self.cap = cv2.VideoCapture(source)
        if width:
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        if height:
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.queue = Queue(maxsize=queue_size)
        self.stopped = False
        t = Thread(target=self._reader, daemon=True)
        t.start()

    def _reader(self):
        while not self.stopped:
            ret, frame = self.cap.read()
            if not ret:
                self.stop()
                break
            if self.queue.full():
                try:
                    self.queue.get_nowait()
                except Empty:
                    pass
            self.queue.put(frame)

    def read(self):
        try:
            return self.queue.get_nowait()
        except Empty:
            return None

    def stop(self):
        self.stopped = True
        try:
            self.cap.release()
        except Exception:
            pass


In [5]:
# %%
# TRACKER (Optional OC-SORT)
TRACKER = None
if CFG['USE_TRACKER']:
    try:
        from ocsort.ocsort import OCSort
        TRACKER = OCSort()
        print('OC-SORT tracker loaded')
    except Exception as e:
        print('OC-SORT import failed — tracking disabled', e)
        TRACKER = None


OC-SORT import failed — tracking disabled No module named 'ocsort'


In [6]:
# %%
# DRAWING UTILITIES

def draw_detections(frame, detections, tracker_out=None, names=None):
    for det in detections:
        x1,y1,x2,y2,conf,cls = det
        x1, y1, x2, y2 = map(int, (x1,y1,x2,y2))
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        label = f"{names[int(cls)] if names else int(cls)} {conf:.2f}"
        cv2.putText(frame, label, (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
    if tracker_out:
        for t in tracker_out:
            tid = int(t[4]) if len(t) > 4 else -1
            bx = list(map(int, t[:4]))
            cv2.putText(frame, f'ID:{tid}', (bx[0], bx[3]+14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)
    return frame


In [10]:
# %%
# INFERENCE LOOP (batching + minimal overhead)

def run_inference(cfg, model, source=0):
    stream = FastVideoCapture(source, queue_size=8)
    time.sleep(0.5)
    batch_size = max(1, int(cfg['BATCH']))
    imgsz = int(cfg['IMG_SIZE'])
    conf = float(cfg['CONF'])
    iou = float(cfg['IOU'])
    frame_buffer = []
    frames_for_draw = []
    last_time = time.time()
    fps_count = 0
    fps = 0.0
    try:
        while True:
            frame = stream.read()
            if frame is None:
                time.sleep(0.005)
                continue
            frames_for_draw.append(frame)
            frame_buffer.append(frame)
            if len(frame_buffer) >= batch_size:
                results = model.predict(frame_buffer, imgsz=imgsz, conf=conf, iou=iou, device=cfg['DEVICE'], verbose=False)
                for res_idx, res in enumerate(results):
                    boxes_list = []
                    if hasattr(res, 'boxes') and len(res.boxes) > 0:
                        xyxy = res.boxes.xyxy.cpu().numpy()
                        confs = res.boxes.conf.cpu().numpy()
                        clss = res.boxes.cls.cpu().numpy()
                        for b, c, cl in zip(xyxy, confs, clss):
                            x1,y1,x2,y2 = b.tolist()
                            boxes_list.append([x1,y1,x2,y2,float(c),int(cl)])
                    tracker_out = None
                    if TRACKER and len(boxes_list) > 0:
                        dets_np = np.array([[d[0],d[1],d[2],d[3],d[4]] for d in boxes_list])
                        if dets_np.size > 0:
                            online_targets = TRACKER.update(dets_np)
                            tracker_out = [t.tolist() for t in online_targets]
                    out_frame = frames_for_draw.pop(0)
                    out_frame = draw_detections(out_frame, boxes_list, tracker_out, names=NAMES)
                    fps_count += 1
                    if fps_count >= 10:
                        now = time.time()
                        fps = fps_count / (now - last_time)
                        last_time = now
                        fps_count = 0
                    if cfg['SHOW']:
                        cv2.putText(out_frame, f'FPS: {fps:.1f}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
                        cv2.imshow('YOLOv8m - Optimized', out_frame)
                frame_buffer = []
                if cfg['SHOW']:
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord('q'):
                        break
    except KeyboardInterrupt:
        print('Interrupted by user')
    finally:
        stream.stop()
        if cfg['SHOW']:
            cv2.destroyAllWindows()

# Run example (uncomment to run):
run_inference(CFG, model, source=CFG['SOURCE'])


### End of appended optimized cells
If you want these injected directly into specific locations in the original notebook,
tell me which cell indices to replace.